<a href="https://colab.research.google.com/github/JithinBinoy-sudo/Seq2Seq-vs-Transformers/blob/main/ML4NLU_Term_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLU Term Paper — Controlled Comparison of Seq2Seq with Attention vs Transformer
## Abstractive Summarization on CNN/DailyMail
**Authors:** Jithin Binoy (1757192) | Keerthana Jagadesh (1774724)

---

## PHASE 1 — Setup & Environment

### 1.1 — Mount Google Drive & Install Dependencies

In [ ]:
# Use local Colab storage (no Drive mount needed)
import os

DRIVE_BASE = "/content/ML4NLU_Project"

# Create subdirectories
for subdir in ["data/raw", "data/processed", "checkpoints/seq2seq", "checkpoints/transformer", "logs", "outputs", "plots"]:
    os.makedirs(os.path.join(DRIVE_BASE, subdir), exist_ok=True)

print(f"Project directory: {DRIVE_BASE}")
print("Subdirectories created successfully.")


In [ ]:
# Install required packages
!pip install -q datasets sentencepiece rouge-score evaluate bert-score torch torchvision matplotlib seaborn tqdm pandas

### 1.2 — Import Libraries & Set Seeds

In [ ]:
import random
import time
import math
import json
import csv
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from torch.cuda.amp import GradScaler

import sentencepiece as spm
from datasets import load_dataset
import evaluate

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ──
assert torch.cuda.is_available(), "GPU not available! Go to Runtime > Change runtime type > GPU."
DEVICE = torch.device('cuda')
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

---
## PHASE 2 — Data Pipeline

### 2.1 — Download CNN/DailyMail Dataset

In [ ]:
# Load CNN/DailyMail v3.0.0
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(f"Train:      {len(dataset['train']):,} articles")
print(f"Validation: {len(dataset['validation']):,} articles")
print(f"Test:       {len(dataset['test']):,} articles")

# Quick sample
sample = dataset['train'][0]
print(f"\nSample article (first 300 chars):\n{sample['article'][:300]}...")
print(f"\nSample highlights:\n{sample['highlights']}")

### 2.2 — Subsample Dataset

In [ ]:
# ── Subsample sizes ──
TRAIN_SIZE = 20_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000

# ── Low-resource subsets for RQ5 ──
LOW_RESOURCE_FRACTIONS = {
    '10pct': int(TRAIN_SIZE * 0.10),   # 5,000
    '25pct': int(TRAIN_SIZE * 0.25),   # 12,500
    '50pct': int(TRAIN_SIZE * 0.50),   # 25,000
}

# Subsample with fixed seed
train_sub = dataset['train'].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_sub   = dataset['validation'].shuffle(seed=SEED).select(range(VAL_SIZE))
test_sub  = dataset['test'].shuffle(seed=SEED).select(range(TEST_SIZE))

# Create low-resource subsets (subsets of the train subsample)
low_resource_subsets = {}
for name, size in LOW_RESOURCE_FRACTIONS.items():
    low_resource_subsets[name] = train_sub.select(range(size))

print(f"Train subset:     {len(train_sub):,}")
print(f"Val subset:       {len(val_sub):,}")
print(f"Test subset:      {len(test_sub):,}")
for name, subset in low_resource_subsets.items():
    print(f"Low-resource {name}: {len(subset):,}")

# Save indices for reproducibility
indices_path = os.path.join(DRIVE_BASE, 'data', 'subsample_indices.json')
indices_data = {
    'train_indices': list(range(TRAIN_SIZE)),
    'val_indices': list(range(VAL_SIZE)),
    'test_indices': list(range(TEST_SIZE)),
    'low_resource': {k: list(range(v)) for k, v in LOW_RESOURCE_FRACTIONS.items()}
}
with open(indices_path, 'w') as f:
    json.dump(indices_data, f)
print(f"\nIndices saved to {indices_path}")

### 2.3 — Train BPE Tokenizer

In [ ]:
VOCAB_SIZE = 50_000
SPM_MODEL_PATH = os.path.join(DRIVE_BASE, 'data/processed', 'spm_bpe')
SPM_MODEL_FILE = SPM_MODEL_PATH + '.model'

if os.path.exists(SPM_MODEL_FILE):
    print(f"Tokenizer already exists at {SPM_MODEL_FILE}, skipping training.")
else:
    # Extract text for tokenizer training
    print("Extracting text for tokenizer training...")
    train_text_path = '/tmp/train_text_for_spm.txt'
    with open(train_text_path, 'w', encoding='utf-8') as f:
        for example in tqdm(train_sub, desc="Extracting text"):
            text = example['article'].lower().strip()
            highlights = example['highlights'].lower().strip()
            f.write(text + '\n')
            f.write(highlights + '\n')

    # Train SentencePiece BPE tokenizer
    print(f"Training BPE tokenizer with vocab_size={VOCAB_SIZE}...")
    spm.SentencePieceTrainer.train(
        input=train_text_path,
        model_prefix=SPM_MODEL_PATH,
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece='<pad>',
        unk_piece='<unk>',
        bos_piece='<bos>',
        eos_piece='<eos>',
        num_threads=os.cpu_count(),
    )
    os.remove(train_text_path)
    print(f"Tokenizer saved to {SPM_MODEL_FILE}")

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(SPM_MODEL_FILE)

# Special token IDs
PAD_ID = sp.pad_id()   # 0
UNK_ID = sp.unk_id()   # 1
BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3

print(f"\nVocab size: {sp.get_piece_size()}")
print(f"PAD={PAD_ID}, UNK={UNK_ID}, BOS={BOS_ID}, EOS={EOS_ID}")

# Verify round-trip
test_text = "the quick brown fox jumps over the lazy dog"
encoded = sp.encode(test_text, out_type=int)
decoded = sp.decode(encoded)
print(f"\nRound-trip test:")
print(f"  Original: {test_text}")
print(f"  Encoded:  {encoded[:20]}...")
print(f"  Decoded:  {decoded}")

### 2.4 — Tokenize & Cache Dataset

In [ ]:
# ── Constants ──
SRC_MAX_LEN = 256
TGT_MAX_LEN = 128

def tokenize_dataset(dataset_split, sp_model, src_max=SRC_MAX_LEN, tgt_max=TGT_MAX_LEN):
    """Tokenize a dataset split into source and target tensors."""
    src_ids_list, tgt_ids_list = [], []
    for example in tqdm(dataset_split, desc="Tokenizing"):
        article = example['article'].lower().strip()
        highlights = example['highlights'].lower().strip()

        src_tokens = sp_model.encode(article, out_type=int)[:src_max]
        tgt_tokens = [BOS_ID] + sp_model.encode(highlights, out_type=int)[:tgt_max - 2] + [EOS_ID]

        src_ids_list.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_ids_list.append(torch.tensor(tgt_tokens, dtype=torch.long))

    return src_ids_list, tgt_ids_list

class SummarizationDataset(Dataset):
    """Dataset for summarization with dynamic padding."""
    def __init__(self, src_ids_list, tgt_ids_list):
        self.src = src_ids_list
        self.tgt = tgt_ids_list

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

def collate_fn(batch):
    """Dynamic padding collate function."""
    src_batch, tgt_batch = zip(*batch)

    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)

    src_mask = (src_padded != PAD_ID)
    tgt_mask = (tgt_padded != PAD_ID)

    return src_padded, tgt_padded, src_mask, tgt_mask

In [ ]:
# ── Tokenize and cache all splits ──
CACHE_DIR = os.path.join(DRIVE_BASE, 'data/processed')

cache_files = {
    'train_50k': (train_sub, 'train_50k.pt'),
    'val_5k':    (val_sub,   'val_5k.pt'),
    'test_5k':   (test_sub,  'test_5k.pt'),
}
# Add low-resource subsets
for name, subset in low_resource_subsets.items():
    cache_files[f'train_{name}'] = (subset, f'train_{name}.pt')

tokenized_data = {}
for key, (data_split, filename) in cache_files.items():
    filepath = os.path.join(CACHE_DIR, filename)
    if os.path.exists(filepath):
        print(f"Loading cached {filename}...")
        tokenized_data[key] = torch.load(filepath)
    else:
        print(f"Tokenizing {key} ({len(data_split):,} samples)...")
        src_ids, tgt_ids = tokenize_dataset(data_split, sp)
        tokenized_data[key] = {'src': src_ids, 'tgt': tgt_ids}
        torch.save(tokenized_data[key], filepath)
        print(f"  Saved to {filepath}")

print("\nAll splits tokenized and cached!")

In [ ]:
# ── Create DataLoaders ──
BATCH_SIZE = 48
NUM_WORKERS = 2

train_dataset = SummarizationDataset(tokenized_data['train_50k']['src'], tokenized_data['train_50k']['tgt'])
val_dataset   = SummarizationDataset(tokenized_data['val_5k']['src'],   tokenized_data['val_5k']['tgt'])
test_dataset  = SummarizationDataset(tokenized_data['test_5k']['src'],  tokenized_data['test_5k']['tgt'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify one batch
src, tgt, src_mask, tgt_mask = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  Source: {src.shape}  Target: {tgt.shape}")
print(f"  Source mask: {src_mask.shape}  Target mask: {tgt_mask.shape}")

---
## PHASE 3 — Model Definitions

### 3.1 — Model A: Seq2Seq with Bahdanau Attention

In [ ]:
class Encoder(nn.Module):
    """Bidirectional LSTM Encoder."""
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(
            embed_dim, hidden_size, num_layers=num_layers,
            bidirectional=True, batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        # Project bidirectional hidden/cell to decoder size
        self.fc_hidden = nn.Linear(hidden_size * 2, hidden_size * 2)
        self.fc_cell   = nn.Linear(hidden_size * 2, hidden_size * 2)

    def forward(self, src, src_mask):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))  # (batch, src_len, embed_dim)

        # Pack padded sequences for efficiency
        lengths = src_mask.sum(dim=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        # outputs: (batch, src_len, hidden_size*2)

        # Merge bidirectional hidden states: concat forward and backward for each layer
        # hidden: (num_layers*2, batch, hidden_size) -> (num_layers, batch, hidden_size*2)
        batch_size = hidden.size(1)
        num_layers = hidden.size(0) // 2
        hidden = hidden.view(num_layers, 2, batch_size, -1)
        hidden = torch.cat([hidden[:, 0], hidden[:, 1]], dim=2)  # (num_layers, batch, hidden*2)
        cell = cell.view(num_layers, 2, batch_size, -1)
        cell = torch.cat([cell[:, 0], cell[:, 1]], dim=2)

        hidden = torch.tanh(self.fc_hidden(hidden))
        cell = torch.tanh(self.fc_cell(cell))

        return outputs, hidden, cell


class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention mechanism."""
    def __init__(self, encoder_dim, decoder_dim):
        super().__init__()
        self.W_enc = nn.Linear(encoder_dim, decoder_dim, bias=False)
        self.W_dec = nn.Linear(decoder_dim, decoder_dim, bias=False)
        self.v = nn.Linear(decoder_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        # decoder_hidden: (batch, decoder_dim)
        # encoder_outputs: (batch, src_len, encoder_dim)
        # src_mask: (batch, src_len)

        src_len = encoder_outputs.size(1)
        dec_expanded = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, dec_dim)

        energy = torch.tanh(self.W_enc(encoder_outputs) + self.W_dec(dec_expanded))
        attention = self.v(energy).squeeze(2)  # (batch, src_len)

        # Mask padding positions
        # Handle length mismatch from pack/unpack
        if src_mask.size(-1) != attention.size(-1):
            src_mask = src_mask[:, :attention.size(-1)]
        attention = attention.masked_fill(~src_mask, float('-inf'))
        weights = F.softmax(attention, dim=1)  # (batch, src_len)

        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (batch, enc_dim)
        return context, weights


class Decoder(nn.Module):
    """Unidirectional LSTM Decoder with Bahdanau Attention."""
    def __init__(self, vocab_size, embed_dim, encoder_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.attention = BahdanauAttention(encoder_dim, hidden_size)
        self.rnn = nn.LSTM(
            embed_dim + encoder_dim, hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_size + encoder_dim + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs, src_mask):
        # input_token: (batch,) — single time step
        embedded = self.dropout(self.embedding(input_token.unsqueeze(1)))  # (batch, 1, embed_dim)

        # Attention using top-layer hidden state
        context, attn_weights = self.attention(hidden[-1], encoder_outputs, src_mask)
        # context: (batch, encoder_dim)

        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)  # (batch, 1, embed+enc)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        # output: (batch, 1, hidden_size)

        output = output.squeeze(1)
        prediction = self.fc_out(torch.cat([output, context, embedded.squeeze(1)], dim=1))
        return prediction, hidden, cell, attn_weights


class Seq2SeqModel(nn.Module):
    """Complete Seq2Seq model with Bahdanau Attention."""
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=256, hidden_size=256,
                 num_layers=2, dropout=0.3):
        super().__init__()
        encoder_dim = hidden_size * 2  # bidirectional
        decoder_dim = encoder_dim      # 512
        self.encoder = Encoder(vocab_size, embed_dim, hidden_size, num_layers, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, encoder_dim, decoder_dim, num_layers, dropout)
        self.vocab_size = vocab_size

    def forward(self, src, tgt, src_mask, teacher_forcing_ratio=1.0):
        batch_size, tgt_len = tgt.shape
        outputs = torch.zeros(batch_size, tgt_len, self.vocab_size, device=src.device)

        encoder_outputs, hidden, cell = self.encoder(src, src_mask)
        input_token = tgt[:, 0]  # <bos>

        all_attn_weights = []
        for t in range(1, tgt_len):
            prediction, hidden, cell, attn_weights = self.decoder(
                input_token, hidden, cell, encoder_outputs, src_mask
            )
            outputs[:, t] = prediction
            all_attn_weights.append(attn_weights)

            if random.random() < teacher_forcing_ratio:
                input_token = tgt[:, t]
            else:
                input_token = prediction.argmax(dim=1)

        return outputs, all_attn_weights

# Instantiate and count parameters
seq2seq_model = Seq2SeqModel().to(DEVICE)
seq2seq_params = sum(p.numel() for p in seq2seq_model.parameters() if p.requires_grad)
print(f"Seq2Seq Model Parameters: {seq2seq_params:,}")
print(f"  ({seq2seq_params / 1e6:.1f}M)")
print()
print(seq2seq_model)

### 3.2 — Model B: Transformer Encoder–Decoder

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerModel(nn.Module):
    """Transformer Encoder-Decoder for summarization."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4,
                 d_ff=1024, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size

        # Shared embedding
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='relu'
        )
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='relu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)

        # Output projection
        self.fc_out = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_square_subsequent_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return mask

    def forward(self, src, tgt, src_mask, tgt_mask=None):
        # src: (batch, src_len), tgt: (batch, tgt_len)
        # src_mask: (batch, src_len) — True for valid tokens

        src_key_padding_mask = ~src_mask  # True = ignore for PyTorch transformer
        tgt_key_padding_mask = ~(tgt != PAD_ID)

        tgt_len = tgt.size(1)
        tgt_causal_mask = self._generate_square_subsequent_mask(tgt_len, src.device)

        # Embeddings with positional encoding
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))

        # Encode
        memory = self.transformer_encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

        # Decode
        output = self.transformer_decoder(
            tgt_emb, memory,
            tgt_mask=tgt_causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        logits = self.fc_out(output)  # (batch, tgt_len, vocab_size)
        return logits

# Instantiate and count parameters
transformer_model = TransformerModel().to(DEVICE)
transformer_params = sum(p.numel() for p in transformer_model.parameters() if p.requires_grad)
print(f"Transformer Model Parameters: {transformer_params:,}")
print(f"  ({transformer_params / 1e6:.1f}M)")

# Check parameter alignment
ratio = max(seq2seq_params, transformer_params) / min(seq2seq_params, transformer_params)
print(f"\nParameter ratio: {ratio:.2f}x", "(ALIGNED ✓)" if ratio <= 1.10 else "(MISALIGNED ✗ — adjust dimensions)")
print()
print(transformer_model)

---
## PHASE 4 — Training Infrastructure

### 4.1 — Common Training Utilities

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """Cross-entropy with label smoothing for Transformer."""
    def __init__(self, vocab_size, padding_idx=PAD_ID, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.criterion = nn.KLDivLoss(reduction='none')

    def forward(self, logits, target):
        # logits: (batch*seq_len, vocab_size), target: (batch*seq_len)
        log_probs = F.log_softmax(logits, dim=-1)
        smooth_target = torch.full_like(log_probs, self.smoothing / (self.vocab_size - 2))
        smooth_target.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        smooth_target[:, self.padding_idx] = 0

        mask = (target != self.padding_idx).unsqueeze(1)
        loss = self.criterion(log_probs, smooth_target)
        loss = (loss * mask).sum() / mask.sum()
        return loss


def get_noam_lr(step, d_model=256, warmup=4000):
    """Noam learning rate schedule (Vaswani et al., 2017)."""
    if step == 0:
        step = 1
    return d_model ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))


class NoamScheduler:
    """Wrapper for Noam LR schedule."""
    def __init__(self, optimizer, d_model=256, warmup=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup = warmup
        self._step = 0

    def step(self):
        self._step += 1
        lr = get_noam_lr(self._step, self.d_model, self.warmup)
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def get_lr(self):
        return get_noam_lr(self._step, self.d_model, self.warmup)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, device,
                grad_accum_steps=1, scheduler=None, teacher_forcing_ratio=1.0,
                is_transformer=False):
    """Train for one epoch with mixed precision and gradient accumulation."""
    model.train()
    total_loss = 0
    num_batches = 0
    optimizer.zero_grad()

    for i, (src, tgt, src_mask, tgt_mask) in enumerate(tqdm(loader, desc="Training", leave=False)):
        src, tgt, src_mask = src.to(device), tgt.to(device), src_mask.to(device)

        with autocast("cuda"):
            if is_transformer:
                tgt_input = tgt[:, :-1]
                tgt_label = tgt[:, 1:]
                logits = model(src, tgt_input, src_mask)
                logits = logits.contiguous().view(-1, model.vocab_size)
                tgt_label = tgt_label.contiguous().view(-1)
                loss = criterion(logits, tgt_label)
            else:
                outputs, _ = model(src, tgt, src_mask, teacher_forcing_ratio)
                outputs = outputs[:, 1:].contiguous().view(-1, model.vocab_size)
                tgt_label = tgt[:, 1:].contiguous().view(-1)
                loss = criterion(outputs, tgt_label)

            loss = loss / grad_accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        total_loss += loss.item() * grad_accum_steps
        num_batches += 1

    return total_loss / num_batches


def validate_epoch(model, loader, criterion, device, is_transformer=False):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for src, tgt, src_mask, tgt_mask in tqdm(loader, desc="Validating", leave=False):
            src, tgt, src_mask = src.to(device), tgt.to(device), src_mask.to(device)

            with autocast("cuda"):
                if is_transformer:
                    tgt_input = tgt[:, :-1]
                    tgt_label = tgt[:, 1:]
                    logits = model(src, tgt_input, src_mask)
                    logits = logits.contiguous().view(-1, model.vocab_size)
                    tgt_label = tgt_label.contiguous().view(-1)
                    loss = criterion(logits, tgt_label)
                else:
                    outputs, _ = model(src, tgt, src_mask, teacher_forcing_ratio=0.0)
                    outputs = outputs[:, 1:].contiguous().view(-1, model.vocab_size)
                    tgt_label = tgt[:, 1:].contiguous().view(-1)
                    loss = criterion(outputs, tgt_label)

            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches

In [ ]:
def save_checkpoint(model, optimizer, epoch, val_loss, filepath, scheduler=None):
    """Save model checkpoint to Drive."""
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
    }
    if scheduler is not None and hasattr(scheduler, '_step'):
        ckpt['scheduler_step'] = scheduler._step
    torch.save(ckpt, filepath)
    print(f"  Checkpoint saved: {filepath} (epoch {epoch}, val_loss={val_loss:.4f})")


def load_checkpoint(model, optimizer, filepath, scheduler=None, device='cuda'):
    """Load model checkpoint from Drive."""
    if not os.path.exists(filepath):
        return 0, float('inf')
    ckpt = torch.load(filepath, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler is not None and 'scheduler_step' in ckpt:
        scheduler._step = ckpt['scheduler_step']
    print(f"  Checkpoint loaded: {filepath} (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
    return ckpt['epoch'], ckpt['val_loss']


def decode_tokens(token_ids, sp_model):
    """Decode token IDs to text, stopping at EOS."""
    tokens = []
    for tid in token_ids:
        if tid == EOS_ID:
            break
        if tid not in (PAD_ID, BOS_ID):
            tokens.append(int(tid))
    return sp_model.decode(tokens)


def log_metrics(log_path, epoch, train_loss, val_loss, lr, extra=None):
    """Append metrics to a CSV log file."""
    header_needed = not os.path.exists(log_path)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'lr': lr}
    if extra:
        row.update(extra)
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if header_needed:
            writer.writeheader()
        writer.writerow(row)

### 4.2 — Train Seq2Seq + Attention Model

In [ ]:
# ── Seq2Seq Training Configuration ──
MAX_EPOCHS = 5
EARLY_STOP_PATIENCE = 3
GRAD_ACCUM_STEPS = 2  # Increase to 2 or 4 if OOM

# Re-initialize model for clean training
seq2seq_model = Seq2SeqModel().to(DEVICE)

# Optimizer with ReduceLROnPlateau
seq2seq_optimizer = torch.optim.Adam(seq2seq_model.parameters(), lr=1e-3, betas=(0.9, 0.999))
seq2seq_scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    seq2seq_optimizer, mode='min', patience=2, factor=0.5
)

# Loss: cross-entropy with padding masked
seq2seq_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

# Mixed precision scaler
seq2seq_scaler = GradScaler()

# Checkpoint paths
SEQ2SEQ_BEST_CKPT = os.path.join(DRIVE_BASE, 'checkpoints/seq2seq/best.pt')
SEQ2SEQ_LAST_CKPT = os.path.join(DRIVE_BASE, 'checkpoints/seq2seq/last.pt')
SEQ2SEQ_LOG = os.path.join(DRIVE_BASE, 'logs/seq2seq_training.csv')

# Resume from checkpoint if available
start_epoch, best_val_loss = load_checkpoint(seq2seq_model, seq2seq_optimizer, SEQ2SEQ_LAST_CKPT)

# Teacher forcing schedule: linear decay from 1.0 to 0.6 over 5 epochs
def get_teacher_forcing_ratio(epoch, start=1.0, end=0.6, decay_epochs=5):
    if epoch >= decay_epochs:
        return end
    return start - (start - end) * epoch / decay_epochs

print(f"Starting Seq2Seq training from epoch {start_epoch}")
print(f"Best val loss so far: {best_val_loss:.4f}")

In [ ]:
# ── Seq2Seq Training Loop ──
seq2seq_train_time_start = time.time()
patience_counter = 0

for epoch in range(start_epoch, MAX_EPOCHS):
    tf_ratio = get_teacher_forcing_ratio(epoch)
    current_lr = seq2seq_optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS} | TF ratio: {tf_ratio:.2f} | LR: {current_lr:.6f}")

    # Train
    torch.cuda.reset_peak_memory_stats()
    train_loss = train_epoch(
        seq2seq_model, train_loader, seq2seq_optimizer, seq2seq_criterion,
        seq2seq_scaler, DEVICE, GRAD_ACCUM_STEPS,
        teacher_forcing_ratio=tf_ratio, is_transformer=False
    )

    # Validate
    val_loss = validate_epoch(seq2seq_model, val_loader, seq2seq_criterion, DEVICE, is_transformer=False)

    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Peak GPU: {peak_mem:.2f} GB")

    # LR schedule
    seq2seq_scheduler_plateau.step(val_loss)

    # Log metrics
    log_metrics(SEQ2SEQ_LOG, epoch + 1, train_loss, val_loss, current_lr,
                {'teacher_forcing': tf_ratio, 'peak_gpu_gb': peak_mem})

    # Save last checkpoint
    save_checkpoint(seq2seq_model, seq2seq_optimizer, epoch + 1, val_loss, SEQ2SEQ_LAST_CKPT)

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(seq2seq_model, seq2seq_optimizer, epoch + 1, val_loss, SEQ2SEQ_BEST_CKPT)
        patience_counter = 0
    else:
        patience_counter += 1

    # Print sample output every 3 epochs
    if (epoch + 1) % 3 == 0 or epoch == 0:
        seq2seq_model.eval()
        with torch.no_grad():
            sample_src, sample_tgt, sample_src_mask, _ = next(iter(val_loader))
            sample_src = sample_src[:1].to(DEVICE)
            sample_tgt = sample_tgt[:1].to(DEVICE)
            sample_src_mask = sample_src_mask[:1].to(DEVICE)
            outputs, _ = seq2seq_model(sample_src, sample_tgt, sample_src_mask, teacher_forcing_ratio=0.0)
            pred_ids = outputs[0].argmax(dim=-1)
            print(f"  Sample prediction: {decode_tokens(pred_ids, sp)[:200]}...")
            print(f"  Sample reference:  {decode_tokens(sample_tgt[0], sp)[:200]}...")

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

seq2seq_train_time = time.time() - seq2seq_train_time_start
print(f"\nSeq2Seq training complete in {seq2seq_train_time/3600:.2f} hours")
print(f"Best validation loss: {best_val_loss:.4f}")

### 4.3 — Train Transformer Model

In [ ]:
# ── Transformer Training Configuration ──
# Re-initialize model
transformer_model = TransformerModel().to(DEVICE)

# Optimizer + Noam schedule
transformer_optimizer = torch.optim.Adam(
    transformer_model.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9
)
noam_scheduler = NoamScheduler(transformer_optimizer, d_model=256, warmup=4000)

# Loss: label-smoothed cross-entropy
transformer_criterion = LabelSmoothingLoss(VOCAB_SIZE, padding_idx=PAD_ID, smoothing=0.1)

# Mixed precision scaler
transformer_scaler = GradScaler()

# Checkpoint paths
TRANSFORMER_BEST_CKPT = os.path.join(DRIVE_BASE, 'checkpoints/transformer/best.pt')
TRANSFORMER_LAST_CKPT = os.path.join(DRIVE_BASE, 'checkpoints/transformer/last.pt')
TRANSFORMER_LOG = os.path.join(DRIVE_BASE, 'logs/transformer_training.csv')

# Resume from checkpoint
start_epoch_t, best_val_loss_t = load_checkpoint(
    transformer_model, transformer_optimizer, TRANSFORMER_LAST_CKPT, scheduler=noam_scheduler
)

print(f"Starting Transformer training from epoch {start_epoch_t}")
print(f"Best val loss so far: {best_val_loss_t:.4f}")

In [ ]:
# ── Transformer Training Loop ──
transformer_train_time_start = time.time()
patience_counter_t = 0

for epoch in range(start_epoch_t, MAX_EPOCHS):
    current_lr = noam_scheduler.get_lr()
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS} | LR: {current_lr:.8f}")

    # Train
    torch.cuda.reset_peak_memory_stats()
    train_loss = train_epoch(
        transformer_model, train_loader, transformer_optimizer, transformer_criterion,
        transformer_scaler, DEVICE, GRAD_ACCUM_STEPS,
        scheduler=noam_scheduler, is_transformer=True
    )

    # Validate
    val_loss = validate_epoch(
        transformer_model, val_loader, transformer_criterion, DEVICE, is_transformer=True
    )

    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Peak GPU: {peak_mem:.2f} GB")

    # Log metrics
    log_metrics(TRANSFORMER_LOG, epoch + 1, train_loss, val_loss, current_lr,
                {'peak_gpu_gb': peak_mem})

    # Save last checkpoint
    save_checkpoint(transformer_model, transformer_optimizer, epoch + 1, val_loss,
                    TRANSFORMER_LAST_CKPT, scheduler=noam_scheduler)

    # Save best checkpoint
    if val_loss < best_val_loss_t:
        best_val_loss_t = val_loss
        save_checkpoint(transformer_model, transformer_optimizer, epoch + 1, val_loss,
                        TRANSFORMER_BEST_CKPT, scheduler=noam_scheduler)
        patience_counter_t = 0
    else:
        patience_counter_t += 1

    # Print sample output every 3 epochs
    if (epoch + 1) % 3 == 0 or epoch == 0:
        transformer_model.eval()
        with torch.no_grad():
            sample_src, sample_tgt, sample_src_mask, _ = next(iter(val_loader))
            sample_src = sample_src[:1].to(DEVICE)
            sample_tgt = sample_tgt[:1].to(DEVICE)
            sample_src_mask = sample_src_mask[:1].to(DEVICE)
            tgt_input = sample_tgt[:, :-1]
            logits = transformer_model(sample_src, tgt_input, sample_src_mask)
            pred_ids = logits[0].argmax(dim=-1)
            print(f"  Sample prediction: {decode_tokens(pred_ids, sp)[:200]}...")
            print(f"  Sample reference:  {decode_tokens(sample_tgt[0], sp)[:200]}...")

    # Early stopping
    if patience_counter_t >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

transformer_train_time = time.time() - transformer_train_time_start
print(f"\nTransformer training complete in {transformer_train_time/3600:.2f} hours")
print(f"Best validation loss: {best_val_loss_t:.4f}")

---
## PHASE 5 — Evaluation

### 5.1 — Summary Generation (Beam Search)

In [ ]:
def beam_search_decode_seq2seq(model, src, src_mask, sp_model, beam_width=2,
                               max_len=80, length_penalty_alpha=0.6):
    """Beam search decoding for Seq2Seq model."""
    model.eval()
    with torch.no_grad():
        device = src.device
        encoder_outputs, hidden, cell = model.encoder(src, src_mask)

        # Initialize beams: (token_ids, score, hidden, cell, finished)
        beams = [([BOS_ID], 0.0, hidden, cell, False)]

        for step in range(max_len):
            all_candidates = []
            for seq, score, h, c, finished in beams:
                if finished:
                    all_candidates.append((seq, score, h, c, True))
                    continue

                input_token = torch.tensor([seq[-1]], device=device)
                prediction, new_h, new_c, _ = model.decoder(
                    input_token, h, c, encoder_outputs, src_mask
                )
                log_probs = F.log_softmax(prediction[0], dim=-1)

                topk_log_probs, topk_ids = log_probs.topk(beam_width)
                for k in range(beam_width):
                    token_id = topk_ids[k].item()
                    new_score = score + topk_log_probs[k].item()
                    new_seq = seq + [token_id]
                    is_finished = (token_id == EOS_ID)

                    # No repeat trigram blocking
                    if len(new_seq) >= 3:
                        trigram = tuple(new_seq[-3:])
                        trigrams = set()
                        for i in range(len(new_seq) - 3):
                            trigrams.add(tuple(new_seq[i:i+3]))
                        if trigram in trigrams:
                            continue

                    all_candidates.append((new_seq, new_score, new_h, new_c, is_finished))

            # Length-normalized score
            def score_fn(candidate):
                seq, score, _, _, _ = candidate
                length = len(seq) - 1  # exclude BOS
                if length == 0:
                    return score
                return score / (length ** length_penalty_alpha)

            if not all_candidates:
                break
            beams = sorted(all_candidates, key=score_fn, reverse=True)[:beam_width]

            if all(b[4] for b in beams):
                break

        if not beams:
            return [BOS_ID, EOS_ID]
        best_seq = max(beams, key=score_fn)[0]
        return best_seq


def beam_search_decode_transformer(model, src, src_mask, sp_model, beam_width=2,
                                    max_len=80, length_penalty_alpha=0.6):
    """Beam search decoding for Transformer model."""
    model.eval()
    with torch.no_grad():
        device = src.device
        src_key_padding_mask = ~src_mask
        src_emb = model.pos_encoder(model.embedding(src) * math.sqrt(model.d_model))
        memory = model.transformer_encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

        # Initialize beams: (token_ids, score, finished)
        beams = [([BOS_ID], 0.0, False)]

        for step in range(max_len):
            all_candidates = []
            for seq, score, finished in beams:
                if finished:
                    all_candidates.append((seq, score, True))
                    continue

                tgt_tensor = torch.tensor([seq], device=device)
                tgt_len = tgt_tensor.size(1)
                tgt_mask = model._generate_square_subsequent_mask(tgt_len, device)
                tgt_emb = model.pos_encoder(model.embedding(tgt_tensor) * math.sqrt(model.d_model))

                output = model.transformer_decoder(
                    tgt_emb, memory,
                    tgt_mask=tgt_mask,
                    memory_key_padding_mask=src_key_padding_mask
                )
                logits = model.fc_out(output[:, -1, :])
                log_probs = F.log_softmax(logits[0], dim=-1)

                topk_log_probs, topk_ids = log_probs.topk(beam_width)
                for k in range(beam_width):
                    token_id = topk_ids[k].item()
                    new_score = score + topk_log_probs[k].item()
                    new_seq = seq + [token_id]
                    is_finished = (token_id == EOS_ID)

                    # No repeat trigram blocking
                    if len(new_seq) >= 3:
                        trigram = tuple(new_seq[-3:])
                        trigrams = set()
                        for i in range(len(new_seq) - 3):
                            trigrams.add(tuple(new_seq[i:i+3]))
                        if trigram in trigrams:
                            continue

                    all_candidates.append((new_seq, new_score, is_finished))

            def score_fn(candidate):
                seq, score, _ = candidate
                length = len(seq) - 1
                if length == 0:
                    return score
                return score / (length ** length_penalty_alpha)

            if not all_candidates:
                break
            beams = sorted(all_candidates, key=score_fn, reverse=True)[:beam_width]

            if all(b[2] for b in beams):
                break

        if not beams:
            return [BOS_ID, EOS_ID]
        best_seq = max(beams, key=score_fn)[0]
        return best_seq

print("Beam search functions defined.")

In [ ]:
# ── Load best checkpoints ──
seq2seq_model = Seq2SeqModel().to(DEVICE)
seq2seq_optimizer_eval = torch.optim.Adam(seq2seq_model.parameters(), lr=1e-3)
load_checkpoint(seq2seq_model, seq2seq_optimizer_eval, SEQ2SEQ_BEST_CKPT)
seq2seq_model.eval()

transformer_model = TransformerModel().to(DEVICE)
transformer_optimizer_eval = torch.optim.Adam(transformer_model.parameters(), lr=1e-4)
load_checkpoint(transformer_model, transformer_optimizer_eval, TRANSFORMER_BEST_CKPT)
transformer_model.eval()

print("Best checkpoints loaded for both models.")

In [ ]:
# ── Generate summaries for all test samples ──
SEQ2SEQ_PREDS_PATH = os.path.join(DRIVE_BASE, 'outputs/seq2seq_preds.txt')
TRANSFORMER_PREDS_PATH = os.path.join(DRIVE_BASE, 'outputs/transformer_preds.txt')

def generate_all_summaries(model, test_data, decode_fn, sp_model, output_path, model_name):
    """Generate summaries for all test samples and save."""
    if os.path.exists(output_path):
        print(f"Summaries already exist at {output_path}, loading...")
        with open(output_path, 'r', encoding='utf-8') as f:
            return f.read().strip().split('\n')

    predictions = []
    src_list = test_data['src']
    print(f"Generating summaries with {model_name}...")

    for i in tqdm(range(len(src_list)), desc=model_name):
        src = src_list[i].unsqueeze(0).to(DEVICE)
        src_mask = (src != PAD_ID)
        pred_ids = decode_fn(model, src, src_mask, sp_model)
        pred_text = decode_tokens(pred_ids, sp_model)
        predictions.append(pred_text)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(predictions))
    print(f"  Saved {len(predictions)} summaries to {output_path}")
    return predictions

seq2seq_preds = generate_all_summaries(
    seq2seq_model, tokenized_data['test_5k'],
    beam_search_decode_seq2seq, sp, SEQ2SEQ_PREDS_PATH, "Seq2Seq"
)

transformer_preds = generate_all_summaries(
    transformer_model, tokenized_data['test_5k'],
    beam_search_decode_transformer, sp, TRANSFORMER_PREDS_PATH, "Transformer"
)

# Get reference summaries
references = []
for i in range(len(tokenized_data['test_5k']['tgt'])):
    ref_text = decode_tokens(tokenized_data['test_5k']['tgt'][i], sp)
    references.append(ref_text)

print(f"\nGenerated {len(seq2seq_preds)} Seq2Seq predictions")
print(f"Generated {len(transformer_preds)} Transformer predictions")
print(f"References: {len(references)}")

### 5.2 — ROUGE Evaluation

In [ ]:
# ── ROUGE Scores ──
rouge_metric = evaluate.load('rouge')

def compute_rouge(predictions, references):
    """Compute ROUGE-1, ROUGE-2, ROUGE-L scores."""
    results = rouge_metric.compute(predictions=predictions, references=references)
    return {
        'rouge1': results['rouge1'],
        'rouge2': results['rouge2'],
        'rougeL': results['rougeL'],
    }

seq2seq_rouge = compute_rouge(seq2seq_preds, references)
transformer_rouge = compute_rouge(transformer_preds, references)

print("=" * 60)
print(f"{'Metric':<12} {'Seq2Seq':>12} {'Transformer':>12}")
print("=" * 60)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    print(f"{metric:<12} {seq2seq_rouge[metric]:>12.4f} {transformer_rouge[metric]:>12.4f}")
print("=" * 60)

In [ ]:
# ── Bootstrap Confidence Intervals & Significance Tests ──
from scipy import stats as scipy_stats

def bootstrap_ci(predictions, references, metric_fn, n_bootstrap=200, ci=0.95):
    """Compute bootstrap confidence intervals for ROUGE scores."""
    n = len(predictions)
    scores = {m: [] for m in ['rouge1', 'rouge2', 'rougeL']}

    for _ in tqdm(range(n_bootstrap), desc="Bootstrap"):
        indices = np.random.randint(0, n, size=n)
        boot_preds = [predictions[i] for i in indices]
        boot_refs = [references[i] for i in indices]
        result = metric_fn(boot_preds, boot_refs)
        for m in scores:
            scores[m].append(result[m])

    ci_results = {}
    alpha = (1 - ci) / 2
    for m in scores:
        arr = sorted(scores[m])
        lower = arr[int(alpha * n_bootstrap)]
        upper = arr[int((1 - alpha) * n_bootstrap)]
        mean = np.mean(arr)
        ci_results[m] = {'mean': mean, 'lower': lower, 'upper': upper}
    return ci_results

print("Computing 95% confidence intervals (this may take a few minutes)...")
seq2seq_ci = bootstrap_ci(seq2seq_preds, references, compute_rouge)
transformer_ci = bootstrap_ci(transformer_preds, references, compute_rouge)

print("\n" + "=" * 80)
print(f"{'Metric':<10} {'Seq2Seq Mean':>14} {'Seq2Seq 95% CI':>20} {'Transformer Mean':>18} {'Transformer 95% CI':>22}")
print("=" * 80)
for m in ['rouge1', 'rouge2', 'rougeL']:
    s = seq2seq_ci[m]
    t = transformer_ci[m]
    print(f"{m:<10} {s['mean']:>14.4f} [{s['lower']:.4f}, {s['upper']:.4f}]   {t['mean']:>14.4f}   [{t['lower']:.4f}, {t['upper']:.4f}]")
print("=" * 80)

# ── Wilcoxon Signed-Rank Test ──
print("\nStatistical Significance (Wilcoxon signed-rank test):")
rouge_per_sample_s2s = rouge_metric.compute(predictions=seq2seq_preds, references=references, use_aggregator=False)
rouge_per_sample_tf  = rouge_metric.compute(predictions=transformer_preds, references=references, use_aggregator=False)

significance_results = {}
for m in ['rouge1', 'rouge2', 'rougeL']:
    stat, p_value = scipy_stats.wilcoxon(rouge_per_sample_s2s[m], rouge_per_sample_tf[m])
    significant = "YES" if p_value < 0.05 else "NO"
    significance_results[m] = {'statistic': stat, 'p_value': p_value, 'significant': significant}
    print(f"  {m}: p={p_value:.6f} -> {'Significant' if p_value < 0.05 else 'Not significant'}")

# Save results
rouge_results = {
    'seq2seq': seq2seq_ci,
    'transformer': transformer_ci,
    'significance': {m: {'p_value': v['p_value'], 'significant': v['significant']}
                     for m, v in significance_results.items()}
}
rouge_results_path = os.path.join(DRIVE_BASE, 'outputs/rouge_results.json')
with open(rouge_results_path, 'w') as f:
    json.dump(rouge_results, f, indent=2, default=float)
print(f"\nROUGE results saved to {rouge_results_path}")

### 5.3 — BERTScore Evaluation (Optional)

In [ ]:
# ── BERTScore ──
from bert_score import score as bert_score_fn

print("Computing BERTScore (this may take several minutes)...")
P_s2s, R_s2s, F1_s2s = bert_score_fn(seq2seq_preds, references, lang='en', verbose=True)
P_tf,  R_tf,  F1_tf  = bert_score_fn(transformer_preds, references, lang='en', verbose=True)

bertscore_results = {
    'seq2seq': {
        'precision': P_s2s.mean().item(),
        'recall': R_s2s.mean().item(),
        'f1': F1_s2s.mean().item(),
    },
    'transformer': {
        'precision': P_tf.mean().item(),
        'recall': R_tf.mean().item(),
        'f1': F1_tf.mean().item(),
    }
}

print(f"\nBERTScore Results:")
print(f"  Seq2Seq    — P: {bertscore_results['seq2seq']['precision']:.4f}, R: {bertscore_results['seq2seq']['recall']:.4f}, F1: {bertscore_results['seq2seq']['f1']:.4f}")
print(f"  Transformer — P: {bertscore_results['transformer']['precision']:.4f}, R: {bertscore_results['transformer']['recall']:.4f}, F1: {bertscore_results['transformer']['f1']:.4f}")

bertscore_path = os.path.join(DRIVE_BASE, 'outputs/bertscore_results.json')
with open(bertscore_path, 'w') as f:
    json.dump(bertscore_results, f, indent=2)
print(f"BERTScore results saved to {bertscore_path}")

### 5.4 — Efficiency Metrics

In [ ]:
# ── Efficiency Metrics ──
def measure_inference_latency(model, test_data, decode_fn, sp_model, n_samples=20, model_name="Model"):
    """Measure average inference latency."""
    model.eval()
    latencies = []

    for i in tqdm(range(min(n_samples, len(test_data['src']))), desc=f"{model_name} latency"):
        src = test_data['src'][i].unsqueeze(0).to(DEVICE)
        src_mask = (src != PAD_ID)

        torch.cuda.synchronize()
        t0 = time.time()
        _ = decode_fn(model, src, src_mask, sp_model)
        torch.cuda.synchronize()
        t1 = time.time()

        latencies.append((t1 - t0) * 1000)  # ms

    return np.mean(latencies), np.std(latencies)

seq2seq_latency_mean, seq2seq_latency_std = measure_inference_latency(
    seq2seq_model, tokenized_data['test_5k'], beam_search_decode_seq2seq, sp,
    n_samples=20, model_name="Seq2Seq"
)

transformer_latency_mean, transformer_latency_std = measure_inference_latency(
    transformer_model, tokenized_data['test_5k'], beam_search_decode_transformer, sp,
    n_samples=20, model_name="Transformer"
)

efficiency_metrics = {
    'seq2seq': {
        'total_training_time_hours': seq2seq_train_time / 3600,
        'peak_gpu_memory_gb': torch.cuda.max_memory_allocated() / 1e9,
        'avg_inference_latency_ms': seq2seq_latency_mean,
        'inference_latency_std_ms': seq2seq_latency_std,
        'parameter_count': seq2seq_params,
    },
    'transformer': {
        'total_training_time_hours': transformer_train_time / 3600,
        'peak_gpu_memory_gb': torch.cuda.max_memory_allocated() / 1e9,
        'avg_inference_latency_ms': transformer_latency_mean,
        'inference_latency_std_ms': transformer_latency_std,
        'parameter_count': transformer_params,
    }
}

print("\nEfficiency Metrics:")
print(f"  {'Metric':<30} {'Seq2Seq':>15} {'Transformer':>15}")
print(f"  {'─'*60}")
for metric in ['total_training_time_hours', 'peak_gpu_memory_gb', 'avg_inference_latency_ms', 'parameter_count']:
    s = efficiency_metrics['seq2seq'][metric]
    t = efficiency_metrics['transformer'][metric]
    fmt = '.2f' if isinstance(s, float) else ','
    print(f"  {metric:<30} {s:>15{fmt}} {t:>15{fmt}}")

efficiency_path = os.path.join(DRIVE_BASE, 'outputs/efficiency_metrics.json')
with open(efficiency_path, 'w') as f:
    json.dump(efficiency_metrics, f, indent=2, default=float)
print(f"\nEfficiency metrics saved to {efficiency_path}")

---
## PHASE 6 — Analysis & Visualization

### 6.1 — Convergence Plots

In [ ]:
# ── Load training logs and plot convergence curves ──
seq2seq_log_df = pd.read_csv(os.path.join(DRIVE_BASE, 'logs/seq2seq_training.csv'))
transformer_log_df = pd.read_csv(os.path.join(DRIVE_BASE, 'logs/transformer_training.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train loss
axes[0].plot(seq2seq_log_df['epoch'], seq2seq_log_df['train_loss'], 'b-o', label='Seq2Seq Train', markersize=4)
axes[0].plot(transformer_log_df['epoch'], transformer_log_df['train_loss'], 'r-s', label='Transformer Train', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation loss
axes[1].plot(seq2seq_log_df['epoch'], seq2seq_log_df['val_loss'], 'b-o', label='Seq2Seq Val', markersize=4)
axes[1].plot(transformer_log_df['epoch'], transformer_log_df['val_loss'], 'r-s', label='Transformer Val', markersize=4)

# Mark best validation epoch
s2s_best_epoch = seq2seq_log_df.loc[seq2seq_log_df['val_loss'].idxmin(), 'epoch']
s2s_best_val = seq2seq_log_df['val_loss'].min()
tf_best_epoch = transformer_log_df.loc[transformer_log_df['val_loss'].idxmin(), 'epoch']
tf_best_val = transformer_log_df['val_loss'].min()

axes[1].axvline(x=s2s_best_epoch, color='blue', linestyle='--', alpha=0.5, label=f'Seq2Seq best (ep {s2s_best_epoch})')
axes[1].axvline(x=tf_best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Transformer best (ep {tf_best_epoch})')
axes[1].plot(s2s_best_epoch, s2s_best_val, 'b*', markersize=15)
axes[1].plot(tf_best_epoch, tf_best_val, 'r*', markersize=15)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
convergence_path = os.path.join(DRIVE_BASE, 'plots/convergence_curves.png')
plt.savefig(convergence_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {convergence_path}")

### 6.2 — Attention Visualizations

In [ ]:
def plot_attention_heatmap(attention_weights, src_tokens, tgt_tokens, title, save_path):
    """Plot attention heatmap."""
    fig, ax = plt.subplots(figsize=(12, 8))

    # Truncate for readability
    max_src = min(50, len(src_tokens))
    max_tgt = min(30, len(tgt_tokens))
    attn = attention_weights[:max_tgt, :max_src]

    sns.heatmap(attn, xticklabels=src_tokens[:max_src], yticklabels=tgt_tokens[:max_tgt],
                cmap='viridis', ax=ax, cbar_kws={'label': 'Attention Weight'})
    ax.set_xlabel('Source Tokens')
    ax.set_ylabel('Target Tokens')
    ax.set_title(title)
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")


# ── Seq2Seq Attention Heatmaps ──
print("Generating Seq2Seq attention heatmaps...")
seq2seq_model.eval()
for idx in range(5):
    src = tokenized_data['test_5k']['src'][idx].unsqueeze(0).to(DEVICE)
    tgt = tokenized_data['test_5k']['tgt'][idx].unsqueeze(0).to(DEVICE)
    src_mask = (src != PAD_ID)

    with torch.no_grad():
        outputs, attn_weights_list = seq2seq_model(src, tgt, src_mask, teacher_forcing_ratio=0.0)

    # Stack attention weights: (tgt_len-1, src_len)
    attn_matrix = torch.stack([w.squeeze(0) for w in attn_weights_list]).cpu().numpy()

    # Get tokens for labels
    src_tokens = [sp.id_to_piece(int(t)) for t in src[0] if t != PAD_ID]
    tgt_tokens = [sp.id_to_piece(int(t)) for t in tgt[0] if t != PAD_ID]

    save_path = os.path.join(DRIVE_BASE, f'plots/attention_seq2seq_{idx}.png')
    plot_attention_heatmap(attn_matrix, src_tokens, tgt_tokens[1:],
                          f'Seq2Seq Bahdanau Attention — Example {idx+1}', save_path)

In [ ]:
# ── Transformer Cross-Attention Heatmaps ──
print("Generating Transformer cross-attention heatmaps...")

# Hook to capture cross-attention weights
cross_attention_weights = {}

def get_cross_attention_hook(layer_idx):
    def hook(module, args, kwargs, output):
        # output is (attn_output, attn_weights)
        cross_attention_weights[layer_idx] = output[1]
    return hook

# Monkey-patch the last decoder layer's cross-attention to return weights
last_decoder_layer = transformer_model.transformer_decoder.layers[-1]
orig_forward = last_decoder_layer.multihead_attn.forward

def patched_forward(*args, **kwargs):
    kwargs['need_weights'] = True
    kwargs['average_attn_weights'] = True
    return orig_forward(*args, **kwargs)

last_decoder_layer.multihead_attn.forward = patched_forward
hook_handle = last_decoder_layer.multihead_attn.register_forward_hook(
    get_cross_attention_hook('last'), with_kwargs=True
)

transformer_model.eval()
for idx in range(5):
    src = tokenized_data['test_5k']['src'][idx].unsqueeze(0).to(DEVICE)
    tgt = tokenized_data['test_5k']['tgt'][idx].unsqueeze(0).to(DEVICE)
    src_mask = (src != PAD_ID)

    cross_attention_weights.clear()
    with torch.no_grad():
        tgt_input = tgt[:, :-1]
        _ = transformer_model(src, tgt_input, src_mask)

    attn_matrix = cross_attention_weights['last'].squeeze(0).cpu().numpy()

    src_tokens = [sp.id_to_piece(int(t)) for t in src[0] if t != PAD_ID]
    tgt_tokens = [sp.id_to_piece(int(t)) for t in tgt[0, :-1] if t != PAD_ID]

    save_path = os.path.join(DRIVE_BASE, f'plots/attention_transformer_{idx}.png')
    plot_attention_heatmap(attn_matrix, src_tokens, tgt_tokens,
                          f'Transformer Cross-Attention (Last Layer) — Example {idx+1}', save_path)

# Restore original forward and remove hook
last_decoder_layer.multihead_attn.forward = orig_forward
hook_handle.remove()
print("Attention visualizations complete.")


In [ ]:
import gc
try: seq2seq_model.cpu()
except: pass
try: transformer_model.cpu()
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

### 6.3 — Low-Resource Experiments (RQ5)

In [ ]:
# ── Low-Resource Training & Evaluation ──

LOW_RESOURCE_EPOCHS = 5
LR_BATCH_SIZE = 16  # Smaller batch for low-resource to avoid OOM
low_resource_results = {'seq2seq': {}, 'transformer': {}}

for subset_name, subset_size in LOW_RESOURCE_FRACTIONS.items():
    print(f"\n{'='*60}")
    print(f"Low-Resource Experiment: {subset_name} ({subset_size:,} samples)")
    print(f"{'='*60}")

    # Create DataLoader for this subset
    lr_dataset = SummarizationDataset(
        tokenized_data[f'train_{subset_name}']['src'],
        tokenized_data[f'train_{subset_name}']['tgt']
    )
    lr_loader = DataLoader(lr_dataset, batch_size=LR_BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

    # ── Train Seq2Seq on this subset ──
    print(f"\nTraining Seq2Seq on {subset_name}...")
    lr_s2s = Seq2SeqModel().to(DEVICE)
    lr_s2s_opt = torch.optim.Adam(lr_s2s.parameters(), lr=1e-3)
    lr_s2s_scaler = GradScaler()
    lr_s2s_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    for epoch in range(LOW_RESOURCE_EPOCHS):
        tf_ratio = get_teacher_forcing_ratio(epoch)
        train_loss = train_epoch(lr_s2s, lr_loader, lr_s2s_opt, lr_s2s_criterion,
                                 lr_s2s_scaler, DEVICE, teacher_forcing_ratio=tf_ratio)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: loss={train_loss:.4f}")

    # Evaluate on test set
    lr_s2s.eval()
    lr_preds = []
    for i in range(min(200, len(tokenized_data['test_5k']['src']))):
        src = tokenized_data['test_5k']['src'][i].unsqueeze(0).to(DEVICE)
        src_mask = (src != PAD_ID)
        pred_ids = beam_search_decode_seq2seq(lr_s2s, src, src_mask, sp)
        lr_preds.append(decode_tokens(pred_ids, sp))
    lr_rouge = compute_rouge(lr_preds, references[:len(lr_preds)])
    low_resource_results['seq2seq'][subset_name] = lr_rouge
    print(f"  Seq2Seq ROUGE-L: {lr_rouge['rougeL']:.4f}")

    del lr_s2s, lr_s2s_opt, lr_s2s_scaler
    torch.cuda.empty_cache()

    # ── Train Transformer on this subset ──
    print(f"\nTraining Transformer on {subset_name}...")
    lr_tf = TransformerModel().to(DEVICE)
    lr_tf_opt = torch.optim.Adam(lr_tf.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9)
    lr_tf_sched = NoamScheduler(lr_tf_opt, d_model=256, warmup=4000)
    lr_tf_scaler = GradScaler()
    lr_tf_criterion = LabelSmoothingLoss(VOCAB_SIZE, padding_idx=PAD_ID, smoothing=0.1)

    for epoch in range(LOW_RESOURCE_EPOCHS):
        train_loss = train_epoch(lr_tf, lr_loader, lr_tf_opt, lr_tf_criterion,
                                 lr_tf_scaler, DEVICE, scheduler=lr_tf_sched, is_transformer=True)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: loss={train_loss:.4f}")

    lr_tf.eval()
    lr_preds_tf = []
    for i in range(min(200, len(tokenized_data['test_5k']['src']))):
        src = tokenized_data['test_5k']['src'][i].unsqueeze(0).to(DEVICE)
        src_mask = (src != PAD_ID)
        pred_ids = beam_search_decode_transformer(lr_tf, src, src_mask, sp)
        lr_preds_tf.append(decode_tokens(pred_ids, sp))
    lr_rouge_tf = compute_rouge(lr_preds_tf, references[:len(lr_preds_tf)])
    low_resource_results['transformer'][subset_name] = lr_rouge_tf
    print(f"  Transformer ROUGE-L: {lr_rouge_tf['rougeL']:.4f}")

    del lr_tf, lr_tf_opt, lr_tf_sched, lr_tf_scaler
    torch.cuda.empty_cache()

# Add full 50K results
low_resource_results['seq2seq']['100pct'] = seq2seq_rouge
low_resource_results['transformer']['100pct'] = transformer_rouge

print("\nLow-resource experiments complete!")

In [ ]:
# ── Plot Low-Resource Curves ──
data_sizes = [5000, 12500, 25000, 50000]
subset_keys = ['10pct', '25pct', '50pct', '100pct']
labels = ['5K (10%)', '12.5K (25%)', '25K (50%)', '50K (100%)']

s2s_rougeL = [low_resource_results['seq2seq'][k]['rougeL'] for k in subset_keys]
tf_rougeL  = [low_resource_results['transformer'][k]['rougeL'] for k in subset_keys]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(data_sizes, s2s_rougeL, 'b-o', label='Seq2Seq + Attention', linewidth=2, markersize=8)
ax.plot(data_sizes, tf_rougeL,  'r-s', label='Transformer', linewidth=2, markersize=8)

ax.set_xlabel('Training Data Size', fontsize=12)
ax.set_ylabel('ROUGE-L', fontsize=12)
ax.set_title('Low-Resource Performance: ROUGE-L vs Training Data Size', fontsize=14)
ax.set_xticks(data_sizes)
ax.set_xticklabels(labels)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
lr_plot_path = os.path.join(DRIVE_BASE, 'plots/low_resource_curves.png')
plt.savefig(lr_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {lr_plot_path}")

### 6.4 — Qualitative Analysis

In [ ]:
# ── Sample 50 test examples for qualitative analysis ──
np.random.seed(SEED)
qual_indices = np.random.choice(len(references), size=50, replace=False)

qual_data = []
for i, idx in enumerate(qual_indices):
    src_text = decode_tokens(tokenized_data['test_5k']['src'][idx], sp)
    ref_text = references[idx]
    s2s_text = seq2seq_preds[idx]
    tf_text  = transformer_preds[idx]

    src_len = len(tokenized_data['test_5k']['src'][idx])
    s2s_compression = len(s2s_text.split()) / max(len(src_text.split()), 1)
    tf_compression  = len(tf_text.split()) / max(len(src_text.split()), 1)

    qual_data.append({
        'index': int(idx),
        'source': src_text[:500],
        'reference': ref_text,
        'seq2seq_output': s2s_text,
        'transformer_output': tf_text,
        'seq2seq_fluency': '',       # To be filled manually (1-5)
        'seq2seq_faithfulness': '',   # To be filled manually (1-5)
        'seq2seq_hallucination': '',  # Y or N
        'seq2seq_compression': round(s2s_compression, 3),
        'transformer_fluency': '',
        'transformer_faithfulness': '',
        'transformer_hallucination': '',
        'transformer_compression': round(tf_compression, 3),
    })

qual_df = pd.DataFrame(qual_data)
qual_csv_path = os.path.join(DRIVE_BASE, 'outputs/qualitative_annotations.csv')
qual_df.to_csv(qual_csv_path, index=False)
print(f"Qualitative annotation template saved to {qual_csv_path}")
print(f"\nSample entries:")
print(qual_df[['index', 'reference', 'seq2seq_output', 'transformer_output']].head(3).to_string())

In [ ]:
# ── Inter-Annotator Agreement (after manual annotation) ──
# This cell runs AFTER both team members have filled in the annotation columns
# Load the filled annotations to compute agreement

def compute_iaa(annotations_path):
    """Compute Cohen's kappa for inter-annotator agreement."""
    try:
        df = pd.read_csv(annotations_path)
        # Check if annotations are filled
        required_cols = ['seq2seq_fluency', 'seq2seq_faithfulness']
        if df[required_cols[0]].isna().all():
            print("Annotations not yet filled. Please complete the annotation CSV first.")
            return

        from sklearn.metrics import cohen_kappa_score
        # Assuming two annotators fill separate CSVs, or columns are suffixed _a1 and _a2
        # For now, compute basic stats on single annotator
        for model in ['seq2seq', 'transformer']:
            fluency = df[f'{model}_fluency'].dropna().astype(int)
            faithfulness = df[f'{model}_faithfulness'].dropna().astype(int)
            hallucination = df[f'{model}_hallucination'].dropna()

            print(f"\n{model.upper()} Annotations:")
            print(f"  Fluency:       mean={fluency.mean():.2f}, std={fluency.std():.2f}")
            print(f"  Faithfulness:  mean={faithfulness.mean():.2f}, std={faithfulness.std():.2f}")
            print(f"  Hallucination: {(hallucination == 'Y').sum()} / {len(hallucination)} = {(hallucination == 'Y').mean():.1%}")
    except Exception as e:
        print(f"Could not compute IAA: {e}")
        print("Fill the qualitative_annotations.csv and re-run this cell.")

compute_iaa(qual_csv_path)

### 6.5 — Results Summary Table

In [ ]:
# ── Compile all results into a summary table ──
summary_data = {
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1',
               'Training Time (hrs)', 'Peak GPU Memory (GB)',
               'Avg Inference Latency (ms)', 'Parameter Count'],
    'Seq2Seq + Attention': [
        f"{seq2seq_ci['rouge1']['mean']:.4f} [{seq2seq_ci['rouge1']['lower']:.4f}, {seq2seq_ci['rouge1']['upper']:.4f}]",
        f"{seq2seq_ci['rouge2']['mean']:.4f} [{seq2seq_ci['rouge2']['lower']:.4f}, {seq2seq_ci['rouge2']['upper']:.4f}]",
        f"{seq2seq_ci['rougeL']['mean']:.4f} [{seq2seq_ci['rougeL']['lower']:.4f}, {seq2seq_ci['rougeL']['upper']:.4f}]",
        f"{bertscore_results['seq2seq']['f1']:.4f}",
        f"{efficiency_metrics['seq2seq']['total_training_time_hours']:.2f}",
        f"{efficiency_metrics['seq2seq']['peak_gpu_memory_gb']:.2f}",
        f"{efficiency_metrics['seq2seq']['avg_inference_latency_ms']:.1f} ± {efficiency_metrics['seq2seq']['inference_latency_std_ms']:.1f}",
        f"{efficiency_metrics['seq2seq']['parameter_count']:,}",
    ],
    'Transformer': [
        f"{transformer_ci['rouge1']['mean']:.4f} [{transformer_ci['rouge1']['lower']:.4f}, {transformer_ci['rouge1']['upper']:.4f}]",
        f"{transformer_ci['rouge2']['mean']:.4f} [{transformer_ci['rouge2']['lower']:.4f}, {transformer_ci['rouge2']['upper']:.4f}]",
        f"{transformer_ci['rougeL']['mean']:.4f} [{transformer_ci['rougeL']['lower']:.4f}, {transformer_ci['rougeL']['upper']:.4f}]",
        f"{bertscore_results['transformer']['f1']:.4f}",
        f"{efficiency_metrics['transformer']['total_training_time_hours']:.2f}",
        f"{efficiency_metrics['transformer']['peak_gpu_memory_gb']:.2f}",
        f"{efficiency_metrics['transformer']['avg_inference_latency_ms']:.1f} ± {efficiency_metrics['transformer']['inference_latency_std_ms']:.1f}",
        f"{efficiency_metrics['transformer']['parameter_count']:,}",
    ],
    'Significant?': [
        significance_results['rouge1']['significant'],
        significance_results['rouge2']['significant'],
        significance_results['rougeL']['significant'],
        '—',
        '—',
        '—',
        '—',
        '—',
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 100)
print("RESULTS SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

# Save as CSV too
summary_csv_path = os.path.join(DRIVE_BASE, 'outputs/results_summary.csv')
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSaved to {summary_csv_path}")

---
## PHASE 7 — Final Paper

### 7.2 — Final Review & Submission Checklist

In [ ]:
# ── Submission Checklist ──
checklist = {
    'Both models train end-to-end from fresh Colab session': True,
    f'Seq2Seq parameters: {seq2seq_params:,}': True,
    f'Transformer parameters: {transformer_params:,}': True,
    f'Parameter ratio within 10%: {max(seq2seq_params, transformer_params)/min(seq2seq_params, transformer_params):.2f}x': max(seq2seq_params, transformer_params)/min(seq2seq_params, transformer_params) <= 1.10,
    'ROUGE-1/2/L with 95% CI computed': os.path.exists(os.path.join(DRIVE_BASE, 'outputs/rouge_results.json')),
    'BERTScore computed': os.path.exists(os.path.join(DRIVE_BASE, 'outputs/bertscore_results.json')),
    'Efficiency metrics recorded': os.path.exists(os.path.join(DRIVE_BASE, 'outputs/efficiency_metrics.json')),
    'Convergence plots generated': os.path.exists(os.path.join(DRIVE_BASE, 'plots/convergence_curves.png')),
    'Low-resource plots generated': os.path.exists(os.path.join(DRIVE_BASE, 'plots/low_resource_curves.png')),
    'Attention heatmaps (>=5 per model)': all(
        os.path.exists(os.path.join(DRIVE_BASE, f'plots/attention_seq2seq_{i}.png')) for i in range(5)
    ) and all(
        os.path.exists(os.path.join(DRIVE_BASE, f'plots/attention_transformer_{i}.png')) for i in range(5)
    ),
    'Qualitative annotation CSV created': os.path.exists(os.path.join(DRIVE_BASE, 'outputs/qualitative_annotations.csv')),
    'Results summary table generated': os.path.exists(os.path.join(DRIVE_BASE, 'outputs/results_summary.csv')),
}

print("SUBMISSION CHECKLIST")
print("=" * 70)
all_pass = True
for item, status in checklist.items():
    symbol = "✅" if status else "❌"
    if not status:
        all_pass = False
    print(f"  {symbol} {item}")
print("=" * 70)
print(f"\nOverall: {'ALL CHECKS PASSED ✅' if all_pass else 'SOME CHECKS FAILED ❌'}")
print("\nRemaining manual tasks:")
print("  • Complete qualitative annotation of 50 samples (both annotators)")
print("  • Compute inter-annotator agreement (Cohen's kappa)")
print("  • Write the final paper following the structure above")
print("  • Submit final notebook and paper")

---
## ✅ Notebook Complete

All phases (1–7) have been implemented. Run cells sequentially from the top.

**Key outputs on Google Drive:**
- `checkpoints/` — trained model weights
- `outputs/` — predictions, ROUGE scores, BERTScore, efficiency metrics
- `plots/` — convergence curves, attention heatmaps, low-resource curves
- `logs/` — training metrics CSV files